In [71]:
# Imports
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Jupyter-specific imports
from IPython.display import display, Markdown

# Set environment variable for protobuf
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"



In [72]:

def load_pdfs_from_folder(folder_path):
    documents = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            loader = UnstructuredPDFLoader(os.path.join(folder_path, file))
            docs = loader.load()
            
            # Add source metadata
            for doc in docs:
                doc.metadata["source"] = file
            
            documents.extend(docs)
    return documents
path_to_pdfs = "/Users/shristisrivastava/Desktop/learning/RAG/project_rag/pdfs"
data = load_pdfs_from_folder(path_to_pdfs)

No languages specified, defaulting to English.


In [7]:
# display(Markdown(f"```\n{data[0].page_content}\n```")) #.page_content

In [73]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
chunks = text_splitter.split_documents(data)
print(f"Text split into {len(chunks)} chunks")

Text split into 32 chunks


In [74]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=OllamaEmbeddings(model="nomic-embed-text"),
    collection_name="local-rag"
)
print("Vector database created successfully")

Vector database created successfully


In [75]:
data = vector_db.get()

for i in range(len(data["documents"])):
    print("\n--- Document", i, "---")
    print("ID:", data["ids"][i])
    print("Text:", data["documents"][i])
    print("Metadata:", data["metadatas"][i])


--- Document 0 ---
ID: 95d000ac-5042-44f2-b8bf-37baa49a4cba
Text: 3 2 0 2

g u A 2

] L C . s c [

7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.

Attention Is All You Need

Ashish Vaswani∗ Google Brain avaswani@google.com

Noam Shazeer∗ Google Brain noam@google.com

Niki Parmar∗ Google Research nikip@google.com

Jakob Uszkoreit∗ Google Research usz@google.com

Llion Jones∗ Google Research llion@google.com

Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu

Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com

Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com

Abstract
Metadata: {'source': 'self_attention.pdf'}

--- Document 1 ---
ID: 9003c82f-9620-4f61-bc38-f7e3c6bba53d
Text: Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu

Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com

Illia Polosukhin∗ ‡ illia.p

In [76]:
llm = ChatOllama(model="llama3.2")


In [77]:
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an expert at generating search queries for a vector database.

Your goal is to improve document retrieval by creating multiple diverse versions of the user's question.

Guidelines:
- Generate 4 different queries
- Each query should capture a different perspective:
  - paraphrase the question
  - use related technical terms
  - expand or specify the intent
  - simplify the question if needed
- Avoid repeating the same wording
- Do NOT answer the question
- Keep queries concise and focused

Output:
Provide ONLY the queries, one per line. No numbering, no explanations.

Original question:
{question}
"""
)

# Set up retriever
retriever = MultiQueryRetriever.from_llm(
    vector_db.as_retriever(search_kwargs={"k": 3}),
    llm,
    prompt=QUERY_PROMPT
)

In [88]:
# RAG prompt template
template = """Answer the question based ONLY on the following context:
{context}
Question: {question}
Note: Return a short, concise, on point answer. If the context does not contain the answer, say "I don't know".
"""

prompt = ChatPromptTemplate.from_template(template)

In [89]:
# Create chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [90]:
# chain.get_graph().print_ascii()

In [ ]:
def ask_with_context(question):
    docs = retriever.invoke(question)   # ✅ FIXED
    
    context = [doc.page_content for doc in docs]
    answer = chain.invoke(question)
    
    return {
        "question": question,
        "answer": answer,
        "contexts": context
    }

In [ ]:
ask_with_context("Give the main idea of the paper")

{'question': 'Give the main idea of the paper',
 'answer': 'The main idea of the paper is to introduce multi-head self-attention and its application in the Transformer model for neural machine translation.',
 'contexts': ['3 2 0 2\n\ng u A 2\n\n] L C . s c [\n\n7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a\n\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\nAttention Is All You Need\n\nAshish Vaswani∗ Google Brain avaswani@google.com\n\nNoam Shazeer∗ Google Brain noam@google.com\n\nNiki Parmar∗ Google Research nikip@google.com\n\nJakob Uszkoreit∗ Google Research usz@google.com\n\nLlion Jones∗ Google Research llion@google.com\n\nAidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu\n\nŁukasz Kaiser∗ Google Brain lukaszkaiser@google.com\n\nIllia Polosukhin∗ ‡ illia.polosukhin@gmail.com\n\nAbstract',
  'have\n\ngovernments\n\n2009\n\nAmerican\n\nmajority\n\na\n\ntha

In [93]:
question_list = [
    "What is self-attention in deep learning?",
    "How does the self-attention mechanism work step by step?",
    "What are Query, Key, and Value vectors in self-attention?",
    "Why is self-attention important in the Transformer architecture?",
    "How does self-attention help in understanding context in a sentence?"
]

ground_truth = [
    "Self-attention is a mechanism that allows a model to evaluate the importance of different tokens in a sequence relative to each other, enabling better contextual understanding.",
    "Self-attention works by converting each token into Query, Key, and Value vectors, computing similarity scores between Query and Key, normalizing them using softmax, and using the resulting weights to compute a weighted sum of Value vectors.",
    "Query, Key, and Value vectors are learned representations of input tokens where Queries are used to compare against Keys to determine relevance, and Values contain the actual information used to compute the output.",
    "Self-attention is important in the Transformer architecture because it enables parallel processing, captures long-range dependencies, and improves efficiency compared to sequential models like RNNs.",
    "Self-attention helps in understanding context by assigning higher weights to relevant words in a sentence, allowing the model to correctly interpret relationships such as references between words."
]

In [94]:
from datasets import Dataset

eval_data = []

for q, gt in zip(question_list, ground_truth):
    result = ask_with_context(q)
    
    eval_data.append({
        "question": q,
        "answer": result["answer"],
        "contexts": result["contexts"],
        "ground_truth": gt
    })

dataset = Dataset.from_list(eval_data)

In [95]:
dataset[0]

{'question': 'What is self-attention in deep learning?',
 'answer': 'Self-attention in deep learning is an attention mechanism that relates different positions of a single sequence to compute a representation of the sequence.',
 'contexts': ['To the best of our knowledge, however, the Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequence- aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate self-attention and discuss its advantages over models such as [17, 18] and [9].\n\n3 Model Architecture\n\nMost competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) to a sequence of continuous representations z = (z1,...,zn). Given z, the decoder then generates an output sequence (y1,...,ym) of symbols one element at a time. At each step t

In [96]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=1800,   # seconds (increase this)
    max_retries=3
)

# Wrap LLM
ragas_llm = LangchainLLMWrapper(llm)

# Wrap embeddings (IMPORTANT)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model="nomic-embed-text")
)


# ✅ Correct evaluation
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=ragas_llm,              
    embeddings=ragas_embeddings ,
    run_config=run_config
)

Evaluating:  35%|███▌      | 7/20 [08:09<21:13, 97.94s/it]Exception raised in Job[8]: OutputParserException(Invalid json output: Given a question and an answer, analyze the complexity of each sentence in the answer. Break down each sentence into one or more fully understandable statements. Ensure that no pronouns are used in any statement. Format the outputs in JSON.
Please return the output in a JSON format that complies with the following schema as specified in JSON Schema:
{"properties": {"statements": {"description": "The generated statements", "items": {"type": "string"}, "title": "Statements", "type": "array"}}, "required": ["statements"], "title": "StatementGeneratorOutput", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.

--------EXAMPLES-----------
Example 1
Input: {
    "question": "Who was Albert Einstein and what is he best known for?",
    "answer": "He was a German-born theoretical physicist, widely acknowled

In [97]:
result

{'faithfulness': nan, 'answer_relevancy': 0.7961, 'context_precision': nan, 'context_recall': 0.3500}

In [ ]:
# vector_db.delete_collection()